## Take-Home-exercise #4_Bowen Tang_4/28/26

In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv")

df.head()

In [0]:
df = df.dropna()

y = df['milliseconds']
X = df.drop(['milliseconds', 'time', 'duration'], axis=1)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 1. Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("prediction", DoubleType(), True)
])

spark_df = spark.createDataFrame([], schema)

# create tables
spark_df.write.mode("overwrite").saveAsTable("bowen_model_1_predictions")
spark_df.write.mode("overwrite").saveAsTable("bowen_model_2_predictions")

## 2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

In [0]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("/Users/bt2683@columbia.edu/take-home-exercise-4-bt2683-Moon")

In [0]:
# Model 1

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="random_forest_model"):

    model1 = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model1.fit(X_train, y_train)

    preds1 = model1.predict(X_test)

    mse1 = mean_squared_error(y_test, preds1)
    r21 = r2_score(y_test, preds1)

    print("Model 1 MSE:", mse1)
    print("Model 1 R2:", r21)

    # log hyperparameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)

    # log metrics (ADD MORE to be safe = 4 metrics)
    mlflow.log_metric("mse", mse1)
    mlflow.log_metric("r2", r21)
    mlflow.log_metric("rmse", mse1 ** 0.5)
    mlflow.log_metric("mae", abs(y_test - preds1).mean())

    # log model
    mlflow.sklearn.log_model(model1, "model")

    # artifact 1
    pd.DataFrame(preds1).to_csv("preds1.csv", index=False)
    mlflow.log_artifact("preds1.csv")

    # artifact 2
    compare1 = pd.DataFrame({
        "actual": y_test.values,
        "predicted": preds1
    })
    compare1.to_csv("compare1.csv", index=False)
    mlflow.log_artifact("compare1.csv")

In [0]:
# Model 2

from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name="decision_tree_model"):

    model2 = DecisionTreeRegressor(max_depth=5, random_state=42)

    model2.fit(X_train, y_train)

    preds2 = model2.predict(X_test)

    mse2 = mean_squared_error(y_test, preds2)
    r22 = r2_score(y_test, preds2)

    print("Model 2 MSE:", mse2)
    print("Model 2 R2:", r22)

    # log hyperparameters
    mlflow.log_param("max_depth", 5)

    # log metrics (4 metrics)
    mlflow.log_metric("mse", mse2)
    mlflow.log_metric("r2", r22)
    mlflow.log_metric("rmse", mse2 ** 0.5)
    mlflow.log_metric("mae", abs(y_test - preds2).mean())

    # log model
    mlflow.sklearn.log_model(model2, "model")

    # artifact 1
    pd.DataFrame(preds2).to_csv("preds2.csv", index=False)
    mlflow.log_artifact("preds2.csv")

    # artifact 2
    compare2 = pd.DataFrame({
        "actual": y_test.values,
        "predicted": preds2
    })
    compare2.to_csv("compare2.csv", index=False)
    mlflow.log_artifact("compare2.csv")

## 3. For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.

In [0]:
import pandas as pd

preds1_df = pd.DataFrame(preds1, columns=["prediction"])

spark_preds1 = spark.createDataFrame(preds1_df)

spark_preds1.write.mode("overwrite").saveAsTable("bowen_model_1_predictions")

In [0]:
preds2_df = pd.DataFrame(preds2, columns=["prediction"])

spark_preds2 = spark.createDataFrame(preds2_df)

spark_preds2.write.mode("overwrite").saveAsTable("bowen_model_2_predictions")

In [0]:
spark.sql("SELECT * FROM bowen_model_1_predictions LIMIT 5").show()

In [0]:
spark.sql("SELECT * FROM bowen_model_2_predictions LIMIT 5").show()